# 06_encode_optimize

『밑바닥부터 시작하는 딥러닝 ❻』 실습 코드 — 원본: `ch04/06_encode_optimize.py`

셀을 위에서부터 차례대로 실행하세요.

In [ ]:
import os, sys

# 노트북에는 __file__이 없으므로 pyproject.toml이 있는 폴더(저장소 루트)를 찾아 이동한다
_dir = os.path.abspath('.')
while not os.path.exists(os.path.join(_dir, 'pyproject.toml')) and _dir != os.path.dirname(_dir):
    _dir = os.path.dirname(_dir)
os.chdir(_dir)
if '.' not in sys.path:
    sys.path.append('.')
print('작업 폴더:', os.getcwd())

In [ ]:
from storybot.tokenizer import pretokenize, count_pairs, merge

In [ ]:
import pickle
import regex as re
from tqdm import tqdm

In [ ]:
class BPETokenizer:
    def __init__(self, merge_rules, end_token="<|endoftext|>"):
        self.merge_rules = merge_rules
        self.end_token = end_token
        self.end_token_id = 256 + len(merge_rules)

        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
        self.id_to_bytes[self.end_token_id] = self.end_token.encode("utf-8")

        self.vocab_size = len(self.id_to_bytes)

    @staticmethod
    def load_from(filepath):
        with open(filepath, "rb") as f:
            merge_rules = pickle.load(f)
        return BPETokenizer(merge_rules)

    def _encode_text(self, text):
        ids = list(text.encode("utf-8"))

        def get_merge_priority(pair):
            return self.merge_rules.get(pair, float('inf'))  # 규칙 목록에 없는 ID 쌍은 우선순위를 가장 낮게 설정

        while len(ids) > 1:
            # 현재 ID열에 있는 인접한 ID 쌍들을 가져옴
            counts = count_pairs(ids)

            # 우선순위가 가장 높은 ID 쌍 찾기
            best_pair = min(counts, key=get_merge_priority)

            # 병합할 수 있는지 확인
            if best_pair not in self.merge_rules:
                break

            # 병합
            new_id = self.merge_rules[best_pair]
            ids = merge(ids, best_pair, new_id)

        return ids

    def encode(self, input_text, show_progress=False):
        pattern = '(' + re.escape(self.end_token) + ')'
        texts = re.split(pattern, input_text)
        all_ids = []

        texts = tqdm(texts) if show_progress else texts

        for text in texts:
            if text == self.end_token:
                all_ids.append(self.end_token_id)
            else:
                for pretoken in pretokenize(text):
                    ids = self._encode_text(pretoken)
                    all_ids.extend(ids)

        return all_ids

    def decode(self, ids):
        byte_list = [self.id_to_bytes[i] for i in ids]
        text_bytes = b"".join(byte_list)
        text = text_bytes.decode("utf-8", errors="replace")
        return text

In [ ]:
if __name__ == "__main__":
    tokenizer = BPETokenizer.load_from("codebot/merge_rules.pkl")

    file_path = "codebot/tiny_codes.txt"
    text = open(file_path).read()
    ids = tokenizer.encode(text, show_progress=True)
    print(len(ids))